## analysis of data from whitehead dataset

Includes data from the following two papers:
1. Kirby MB^, Petersen BM^, Faris J, Kells S^, Sprenger K, Whitehead TA* “Retrospective SARS-CoV-2 antibody development trajectories are sparse and permissive”. Proceedings National Academy of Sciences in press (2024)
2. Petersen BM^, Kirby MB^, Chrispens KM&, Irvin OI^, Strawn IK^, Haas CM&, Walker AM&, Baumer ZT^, Ulmer SA&, Ayala E, Rhodes ER, Guthmiller JJ, Steiner PJ^, Whitehead TA* “An integrated technology for quantitative wide mutational scanning of human antibody Fab libraries” Nature Communications, (2024) 15 (1) (May 10, 2024): ARTN 3974.

File explanation from paper source data:
1. 41467_2024_48072_MOESM9_ESM.xlsx - from paper 1 "Retrospective SARS-CoV-2..."
    a. Figure 4c tab seems to be the one containing the antibodies (in the form of antibody name> mutations), and the matching Kds. However, there's a different number of antibodies in the data from Michael - 1041, and the data in the figure 4c tab - 1303 unique antibodies, 1289 of them have two replicate measurements and the rest have one.
    
2. pnas... files from paper 2 "An integrated technology.."




File explanation from Michael Chungyoun:

Kirby_PNAS2025_FLAB_filtered – this file reports antibody sequences with a float Kd (reported in nanomolar)
Kirby_PNAS2025_FLAB_binary – this file reports antibody sequences with a binary (1 if Kd <=3000, 0 if >3000). We applied the same filtering of the dataset as used in the paper. 

The antigen used for these experiments is SARS-CoV2 Spike Wuhan Hu-1 S1 subunit (positions V16-R685). The Kd values reported are for monovalent dissociation constants (this will likely be a big issue for these datasets, best to report upfront).
 
Here is the reference for the paper to include:

 
Kirby MB^, Petersen BM^, Faris J, Kells S^, Sprenger K, Whitehead TA* “Retrospective SARS-CoV-2 antibody development trajectories are sparse and permissive”. Proceedings National Academy of Sciences in press (2024)
"""

Email2:
"""
Same analysis as before, same outputs. I extracted out the two antibodies with the most useful data. They were tested against influenza HA H1 (The ectodomain of A/Brisbane/02/2018 H1 HA with a foldon trimerization domain was expressed in HEK293T cells (ATCC, female, #CRL-11268) and purified using Ni-NTA affinity chromatography).

Petersen BM^, Kirby MB^, Chrispens KM&, Irvin OI^, Strawn IK^, Haas CM&, Walker AM&, Baumer ZT^, Ulmer SA&, Ayala E, Rhodes ER, Guthmiller JJ, Steiner PJ^, Whitehead TA* “An integrated technology for quantitative wide mutational scanning of human antibody Fab libraries” Nature Communications, (2024) 15 (1) (May 10, 2024): ARTN 3974.

MAGMAseq_anchor_FLABbinary.csv
MAGMAseq_anchor_FLAB.csv

In [ ]:
from collections import Counter
from dataclasses import dataclass
import io
import os
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from tqdm import tqdm
tqdm.pandas()

from netam.framework import load_crepe
from netam.sequences import translate_sequence, AA_STR_SORTED
from dnsmex.ablang_wrapper import AbLangWrapper
from dnsmex.esm_wrapper import esm2_wrapper_of_size
from dnsmex.local import localify

figures_dir = localify("FIGURES_DIR")
os.makedirs("_output", exist_ok=True)

### An integrated technology for quantitative wide mutational scanning of human antibody Fab libraries

In [ ]:
# this is the dataset provdied by Michael Chungyoun
michael_dataset = pd.read_csv(localify("DATA_DIR/whitehead/MAGMAseq_anchor_FLAB.csv"))

# this is the dataset lableled as processed antibody sequecne bindign datasets - supplementary data4
df_4A8_CC121_combined = pd.read_excel(localify("DATA_DIR/whitehead/41467_2024_48072_MOESM6_ESM_supplementary_data4.xlsx"), sheet_name='4A8_CC121_combined')
## Tab	Description
#all_barcodes_4A8_CC121	all barcode estimates using top25 bins for 4A8 and CC12.1 with S1 antigen
#4A8_CC121_combined	all variant estimates for 4A8 and CC12.1 from collapsed barcode counts for top25 and top50 bins with S1 antigen
#all_barcodes_222-1C06	all barcode estimates for 222-1C06 with HA antigen
#all_barcodes_319-345	all barcode estimates for 319-345 with HA antigen
#topbinHA_rep2	all replicate 2 estimates using top25 bins for HA antigen


# this is the source data (data for figures from the paper)
source_data = pd.read_csv(localify("DATA_DIR/whitehead/41467_2024_48072_MOESM6_ESM_supplementary_data4.xlsx"))
